Pytorch includes a scaled dot-product attention operator:

Measures how much a query matches a key. Measures how much each position weights into how the model gets the answer. in Video Masked Auto-Endoders, this runs over spatiotemporal patch tokens, each patch learns how much the other patches are relevant to creating that patch. 

Attention(Q, K, V) = softmax(QK^T / √d_k) V

- QK^T: dot product between queiris and keys, measures similarity
- / √d_k: scales down dot product by key dimension to prevent exploding softmax
- softmax: converts scores to a probability distribution
- x V: weighted sum of values

from transformers import VideoMAEForVideoClassification

model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics", attn_implementation="sdpa", dtype=torch.float16)

Notes for my implementation
- Each video must have the same number of frames
- Once setup, there is lots to experiment with how to deal with behavior imbalance, how to train the VideoMAE model (freeze MAE, tune classifier for ex.)
- Also need to get to data augmentation
- Consider freezing first N layers in YOLO to help avoid overfitting when training on small data sets (such as mine)

TODO:
- Get this setup to run VideoMAE, see the confusion matrix
- Move pipeline temps and tasks into more formalized pipeline files with config classes
- APPLY while experimenting with data augmentation, imbalanced sampling, training, models, etc.
    - Hopefully won't be so bad to iterate once this full workflow is set up
- We should also consider building a pipeline that will predict on a batch of videos given
    - camera_id, paths, trained models, etc.

In [3]:
from transformers import VideoMAEConfig, VideoMAEModel

# Initialize VideoMAE config
configuration = VideoMAEConfig()

model = VideoMAEModel(config=configuration)

# Can access model configuration
configuration = model.config

In [4]:
# Can reate a VideoMAE image processor
# VideoMAEImageProcessor - effectively loads videos into the model
# -> be sure to check what the pixels are normalized to (0-255 vs 0-1 will require different setting)
# -> returns tensors or stacked tensors
# --> tensors can run on the GPU and trace operations done to them (think backward())

# VideoMAEImageProcessorPil

# VideoMAEVideoProcessor

# ... I'm not really sure what's different between these processors

In [5]:
# VideoMAEModel
# -> more basic model

In [ ]:
# VideoMAEForPreTraining
# -> Includes the decoder on top for self-supervised pre-training
# -> Its output if thd reconstructed pixel values for the masked out tube patches
# -> WOuld would use this if you want to do your own pre-training, see if you can improve this part

In [ ]:
# VideoMAEForVideoClassification
# -> Can retrain full model (MAE and classification layers)
# -> Retraining the full model can create overfitting though
#    -> recommended to sart by onnly tuning the classification layers
#       then if there is plateauing, start to unfreeze last few MAE layers